# MedSigLIP Feature Extraction for Medical Images

This notebook demonstrates how to use the MedSigLIP model for feature extraction from chest X-ray images. MedSigLIP is a medical vision-language model that can extract embeddings and perform zero-shot classification on medical images.

## 1. Setup Environment and Install Dependencies

First, ensure all required packages are installed.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers torch pillow requests numpy

## 2. Configure Hugging Face Access

Set up your Hugging Face access token to authenticate with the model hub. You can get your token from [Hugging Face Settings](https://huggingface.co/settings/tokens).

In [ ]:
import os

# Set your Hugging Face token (choose one method):

# Method 1: Set directly in notebook (not recommended for production)
# os.environ["HF_TOKEN"] = "your_token_here"

# Method 2: Load from .env file (recommended)
# from dotenv import load_dotenv
# load_dotenv()

# Method 3: Already set in system environment variables
# The token will be automatically loaded from HF_TOKEN, HUGGINGFACE_TOKEN, or HF_ACCESS_TOKEN

# Verify token is set
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_ACCESS_TOKEN")
if hf_token:
    print("✓ Hugging Face token is configured")
else:
    print("⚠ Warning: No Hugging Face token found. Set HF_TOKEN environment variable.")

## 3. Import Required Libraries

In [ ]:
import numpy as np
from PIL import Image
import requests
from transformers import AutoProcessor, AutoModel
import torch
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 4. Initialize Model and Processor

Load the MedSigLIP-448 model from Hugging Face and configure the device.

In [ ]:
# Configure device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model and processor
print("Loading MedSigLIP model...")
model = AutoModel.from_pretrained("google/medsiglip-448", token=hf_token).to(device)
processor = AutoProcessor.from_pretrained("google/medsiglip-448", token=hf_token)
model.eval()
print("✓ Model loaded successfully")

## 5. Download Sample Images

Download sample medical images for testing.

In [ ]:
# Download sample images
!wget -nc -q https://storage.googleapis.com/dx-scin-public-data/dataset/images/3445096909671059178.png
!wget -nc -q https://storage.googleapis.com/dx-scin-public-data/dataset/images/-5669089898008966381.png

print("✓ Sample images downloaded")

## 6. Define Image Resizing Function

Create a function to resize images to 448×448 using Pillow's bilinear interpolation.

In [ ]:
def resize(image):
    """
    Resize image to 448x448 using Pillow's bilinear interpolation.
    This provides similar results to the Big Vision library implementation.
    """
    return image.resize((448, 448), Image.Resampling.BILINEAR)

## 7. Prepare Images for Processing

Load and resize the images.

In [ ]:
# Load images
imgs = [
    Image.open("3445096909671059178.png").convert("RGB"),
    Image.open("-5669089898008966381.png").convert("RGB")
]

# Resize images
resized_imgs = [resize(img) for img in imgs]

print(f"✓ Loaded and resized {len(imgs)} images")
print(f"  Image shapes: {[img.size for img in resized_imgs]}")

## 8. Define Text Labels

Create text descriptions for zero-shot classification.

In [ ]:
# Define text labels for classification
texts = [
    "a photo of an arm with no rash",
    "a photo of an arm with a rash",
    "a photo of a leg with no rash",
    "a photo of a leg with a rash"
]

print(f"Text labels: {len(texts)}")
for i, text in enumerate(texts, 1):
    print(f"  {i}. {text}")

## 9. Process Inputs and Extract Features

Pass images and text through the model to extract embeddings.

In [ ]:
# Prepare inputs
inputs = processor(text=texts, images=resized_imgs, padding="max_length", return_tensors="pt").to(device)

# Extract features
print("Extracting features...")
with torch.no_grad():
    outputs = model(**inputs)

print("✓ Feature extraction complete")
print(f"  Image embeddings shape: {outputs.image_embeds.shape}")
print(f"  Text embeddings shape: {outputs.text_embeds.shape}")

## 10. Calculate Probabilities

Compute softmax probabilities for image-text matching.

In [ ]:
# Calculate probabilities
logits_per_image = outputs.logits_per_image
probs = torch.softmax(logits_per_image, dim=1)

print("Probability matrix shape:", probs.shape)
print(f"[{len(imgs)} images × {len(texts)} text labels]")

## 11. Display Results

Show each image with its classification probabilities.

In [ ]:
# Display results for each image
for n_img, img in enumerate(imgs):
    print(f"\n{'='*60}")
    print(f"Image {n_img + 1}:")
    print(f"{'='*60}")
    display(img)
    print("\nClassification probabilities:")
    for i, label in enumerate(texts):
        prob = probs[n_img][i]
        bar = '█' * int(prob * 50)
        print(f"{prob:.2%} {bar:50s} '{label}'")

## 12. Extract and Display Embeddings

Show the multimodal embeddings for downstream tasks like similarity search or clustering.

In [ ]:
# Display embedding information
print("\n" + "="*60)
print("EMBEDDINGS")
print("="*60)

print(f"\nImage Embeddings:")
print(f"  Shape: {outputs.image_embeds.shape}")
print(f"  Dtype: {outputs.image_embeds.dtype}")
print(f"  Sample (first 5 dims): {outputs.image_embeds[0, :5].cpu().numpy()}")

print(f"\nText Embeddings:")
print(f"  Shape: {outputs.text_embeds.shape}")
print(f"  Dtype: {outputs.text_embeds.dtype}")
print(f"  Sample (first 5 dims): {outputs.text_embeds[0, :5].cpu().numpy()}")

print("\n✓ Embeddings can be used for:")
print("  • Similarity search")
print("  • Clustering")
print("  • Retrieval tasks")
print("  • Fine-tuning downstream models")

---

## Using the CXR Agent FeatureExtractorAdapter

You can also use the integrated adapter from the CXR Agent:

In [ ]:
# Example using the FeatureExtractorAdapter
import sys
sys.path.append('..')

from src.models.adapters import FeatureExtractorAdapter

# Initialize adapter
extractor = FeatureExtractorAdapter(
    model_name="google/medsiglip-448",
    hf_token=hf_token  # Will auto-load from environment if None
)

# Load model
await extractor.load()

# Extract features from single image
result = await extractor.predict(
    image_path="3445096909671059178.png",
    texts=[
        "chest X-ray showing normal lungs",
        "chest X-ray showing pneumonia",
        "chest X-ray showing cardiomegaly"
    ],
    return_embeddings=True,
    return_similarities=True
)

print("\nAdapter Results:")
print(f"  Extraction time: {result['extraction_time_ms']:.2f}ms")
print(f"  Embedding shape: {result['image_embedding_shape']}")
print(f"\nSimilarities:")
for sim in result['similarities']:
    print(f"  Top match: '{sim['top_match'][0]}' ({sim['top_match'][1]:.2%})")